## Teoría base (cómo leer los bloques de código)

Pipeline de un encoder fine-tuned:

1. Dataset y splits.
2. Tokenización.
3. Modelo + pérdida.
4. Entrenamiento.
5. Evaluación e inferencia.

Objetivo supervisado:
$$
\min_\theta\,\mathbb{E}_{(x,y)}\left[\ell(f_\theta(x),y)\right]
$$

En cada bloque, relaciona el código con una pregunta: ¿qué mejora exactamente la métrica?

## Guía de lectura (teoría ↔ práctica)

- **Objetivo**: resolver una tarea real de clasificación con fine-tuning de encoder.
- **Pipeline matemático**:
$$
x \rightarrow \text{tokenizer} \rightarrow \text{encoder} \rightarrow h_{\text{CLS}} \rightarrow \text{classifier}
$$
$$
\mathcal{L}_{CE} = -\log p_\theta(y\mid x)
$$
- **Qué observar**: impacto de `learning_rate`, `num_train_epochs` y tamaño de dataset en `f1_macro`.

# Ejercicio

Fine-tunear un modelo encoder-only en una tarea de clasificación de texto.
Ejemplos de modelos:
- RoBERTa: https://huggingface.co/FacebookAI/roberta-base (en inglés)
- CodeBERT: https://huggingface.co/microsoft/codebert-base (tareas de clasificación de código)
- BERT: https://huggingface.co/google-bert/bert-base-uncased (bert, en inglés)
- etc.

Ejemplos de datasets:
- AG News: https://huggingface.co/datasets/fancyzhx/ag_news (4 clases, en inglés)
- BigCloneBench: https://huggingface.co/datasets/google/code_x_glue_cc_clone_detection_big_clone_bench (2 clases - misma funcionalidad o no, código). Dataset muy grande considerar coger un subset. Usar `<s>code1</s>code2</s>` como inputs.
- SocialTOX: https://huggingface.co/datasets/gplsi/SocialTOX  (tóxico, ligeramente tóxico y no tóxico, en español)

## Librerías y configuración

Usamos `AG News` (4 clases) y fine-tuning de `bert-base-uncased` con Hugging Face `Trainer`.

In [1]:
import numpy as np
from datasets import load_dataset
import evaluate
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    set_seed,
)

set_seed(42)

## Dataset

Cargamos AG News y creamos split de validación desde train. Además, usamos un subset para que el entrenamiento sea razonablemente rápido en clase.

In [2]:
ds = load_dataset("ag_news")

# train/validation a partir de train
split = ds["train"].train_test_split(test_size=0.1, seed=42)
ds = {
    "train": split["train"],
    "validation": split["test"],
    "test": ds["test"],
}

# Subset para entrenar más rápido (ajusta si quieres más calidad)
max_train = 12000
max_val = 2000
max_test = 2000

ds["train"] = ds["train"].shuffle(seed=42).select(range(min(max_train, len(ds["train"]))))
ds["validation"] = ds["validation"].shuffle(seed=42).select(range(min(max_val, len(ds["validation"]))))
ds["test"] = ds["test"].shuffle(seed=42).select(range(min(max_test, len(ds["test"]))))

for split_name in ["train", "validation", "test"]:
    print(split_name, len(ds[split_name]))

display(ds["train"].shuffle(seed=1).select(range(5)).to_pandas())

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

train 12000
validation 2000
test 2000


,text,label
0,Senior Fatah Official Shot Dead in Nablus (AP)...,0
1,Bahrain Activist Pardoned by King Bahrain #39;...,0
2,Hrbaty Suffers Upset at St. Petersburg (AP) AP...,1
3,Cave Linked to John the Baptist Found in Israe...,3
4,Bayer refit leads to US sales pact Bayer forge...,2


## Tokenización y preprocesado

In [3]:
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
max_length = 256

def preprocess(batch):
    return tokenizer(batch["text"], truncation=True, max_length=max_length)

encoded = {
    "train": ds["train"].map(preprocess, batched=True),
    "validation": ds["validation"].map(preprocess, batched=True),
    "test": ds["test"].map(preprocess, batched=True),
}

for split_name in ["train", "validation", "test"]:
    encoded[split_name] = encoded[split_name].rename_column("label", "labels")
    encoded[split_name] = encoded[split_name].remove_columns(
        [c for c in encoded[split_name].column_names if c not in ["input_ids", "attention_mask", "labels"]]
    )

print(tokenizer.decode(encoded["train"][0]["input_ids"][:40]))
print("Label:", encoded["train"][0]["labels"])

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/12000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

[CLS] carrefour lowers annual profit forecast on price cuts ( update3 ) carrefour sa, the world # 39 ; s second - largest retailer, said annual profit will not meet a forecast for
Label: 2


## Modelo y métricas

In [4]:
label_names = ds["train"].features["label"].names
num_labels = len(label_names)
id2label = {i: label_names[i] for i in range(num_labels)}
label2id = {label_names[i]: i for i in range(num_labels)}

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
)

acc = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": acc.compute(predictions=preds, references=labels)["accuracy"],
        "f1_macro": f1.compute(predictions=preds, references=labels, average="macro")["f1"],
    }

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## Entrenamiento

In [5]:
from transformers import EarlyStoppingCallback

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

args = TrainingArguments(
    output_dir="bert-ag-news-exercise",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=2,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    logging_strategy="epoch",
    report_to="none",
    save_total_limit=2,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=encoded["train"],
    eval_dataset=encoded["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.343846,0.274589,0.907500,0.905715
2,0.167874,0.283638,0.917500,0.915983


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=1500, training_loss=0.25585985310872394, metrics={'train_runtime': 2115.6066, 'train_samples_per_second': 11.344, 'train_steps_per_second': 0.709, 'total_flos': 1206648967520256.0, 'train_loss': 0.25585985310872394, 'epoch': 2.0})

## Evaluación en test e inferencia rápida

In [6]:
from pprint import pprint
from transformers import pipeline

# evaluación en test
test_metrics = trainer.evaluate(encoded["test"])
pprint(test_metrics)

# inferencia de ejemplo
clf = pipeline("text-classification", model=trainer.model, tokenizer=tokenizer)

examples = [
    "The company reported strong quarterly earnings and shares rose.",
    "The team secured a dramatic last-minute victory in the final.",
    "Scientists discovered a new method to improve battery performance.",
    "The government announced changes to trade policy today.",
]

for text in examples:
    pred = clf(text)[0]
    print(f"\nText: {text}")
    print(pred)

{'epoch': 2.0,
 'eval_accuracy': 0.9145,
 'eval_f1_macro': 0.9154749510259735,
 'eval_loss': 0.293346107006073,
 'eval_runtime': 202.6148,
 'eval_samples_per_second': 9.871,
 'eval_steps_per_second': 0.311}

Text: The company reported strong quarterly earnings and shares rose.
{'label': 'Business', 'score': 0.9885318279266357}

Text: The team secured a dramatic last-minute victory in the final.
{'label': 'Sports', 'score': 0.9969052672386169}

Text: Scientists discovered a new method to improve battery performance.
{'label': 'Sci/Tech', 'score': 0.989331841468811}

Text: The government announced changes to trade policy today.
{'label': 'Business', 'score': 0.9871652126312256}
